In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('../bitcoin_data/raw/btc_daily_clean.csv')
data.head()


,date,open,high,low,close,volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4298 entries, 0 to 4297
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    4298 non-null   str    
 1   open    4298 non-null   float64
 2   high    4298 non-null   float64
 3   low     4298 non-null   float64
 4   close   4298 non-null   float64
 5   volume  4298 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 201.6 KB


In [4]:
## take values for 3 decimal
cols = ["open", "high", "low", "close", "volume"]

data[cols] = data[cols].astype(float).round(3)

In [5]:
data.head()

,date,open,high,low,close,volume
0,2014-09-17,465.864,468.174,452.422,457.334,21056800.0
1,2014-09-18,456.860,456.860,413.104,424.440,34483200.0
2,2014-09-19,424.103,427.835,384.532,394.796,37919700.0
3,2014-09-20,394.673,423.296,389.883,408.904,36863600.0
4,2014-09-21,408.085,412.426,393.181,398.821,26580100.0


In [6]:
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values("date").reset_index(drop=True)

In [7]:
df = data.copy()

In [8]:
### Check duplicate values
duplicate_rows = df[df.duplicated()]

print("Number of duplicate rows:", duplicate_rows.shape[0])
print(duplicate_rows)

Number of duplicate rows: 0
Empty DataFrame
Columns: [date, open, high, low, close, volume]
Index: []


##### Feature Engineering

In [9]:
def compute_rsi(close, window=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


def compute_atr(df, window=14):
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift(1)).abs()
    low_close = (df["low"] - df["close"].shift(1)).abs()

    true_range = pd.concat(
        [high_low, high_close, low_close],
        axis=1
    ).max(axis=1)

    atr = true_range.rolling(window).mean()

    return atr


def rolling_zscore(series, window):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()

    return (series - mean) / std

In [10]:
def create_btc_indicators(df):
    df = df.copy()
    df = df.sort_index()

    # ----------------------------
    # Price return indicators
    # ----------------------------
    df["log_close"] = np.log(df["close"])

    for h in [1, 2, 3, 5, 7, 14, 21, 30]:
        df[f"log_return_{h}d"] = df["log_close"].diff(h)
        df[f"pct_return_{h}d"] = df["close"].pct_change(h)

    # ----------------------------
    # Moving averages
    # ----------------------------
    for window in [7, 14, 20, 30, 50, 100, 200]:
        df[f"sma_{window}"] = df["close"].rolling(window).mean()
        df[f"ema_{window}"] = df["close"].ewm(span=window, adjust=False).mean()

        df[f"close_to_sma_{window}"] = df["close"] / df[f"sma_{window}"] - 1
        df[f"close_to_ema_{window}"] = df["close"] / df[f"ema_{window}"] - 1

        df[f"sma_{window}_slope"] = df[f"sma_{window}"].pct_change(5)
        df[f"ema_{window}_slope"] = df[f"ema_{window}"].pct_change(5)

    # Moving average crossover indicators
    df["sma_20_above_50"] = (df["sma_20"] > df["sma_50"]).astype(int)
    df["sma_50_above_200"] = (df["sma_50"] > df["sma_200"]).astype(int)
    df["ema_20_above_50"] = (df["ema_20"] > df["ema_50"]).astype(int)

    # ----------------------------
    # RSI
    # ----------------------------
    df["rsi_14"] = compute_rsi(df["close"], window=14)

    df["rsi_overbought"] = (df["rsi_14"] > 70).astype(int)
    df["rsi_oversold"] = (df["rsi_14"] < 30).astype(int)

    # ----------------------------
    # MACD
    # ----------------------------
    ema_12 = df["close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["close"].ewm(span=26, adjust=False).mean()

    df["macd"] = ema_12 - ema_26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    df["macd_bullish"] = (df["macd"] > df["macd_signal"]).astype(int)

    # ----------------------------
    # Volatility indicators
    # ----------------------------
    for window in [7, 14, 21, 30, 60, 90]:
        df[f"realized_vol_{window}d"] = df["log_return_1d"].rolling(window).std()
        df[f"realized_vol_{window}d_ann"] = df[f"realized_vol_{window}d"] * np.sqrt(365)

    df["atr_14"] = compute_atr(df, window=14)
    df["atr_pct_14"] = df["atr_14"] / df["close"]

    df["daily_range"] = df["high"] - df["low"]
    df["daily_range_pct"] = df["daily_range"] / df["close"]

    df["close_position_in_range"] = (
        (df["close"] - df["low"]) / (df["high"] - df["low"])
    )

    # ----------------------------
    # Bollinger Bands
    # ----------------------------
    bb_window = 20

    df["bb_middle"] = df["close"].rolling(bb_window).mean()
    bb_std = df["close"].rolling(bb_window).std()

    df["bb_upper"] = df["bb_middle"] + 2 * bb_std
    df["bb_lower"] = df["bb_middle"] - 2 * bb_std

    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_middle"]

    df["bb_percent_b"] = (
        (df["close"] - df["bb_lower"]) /
        (df["bb_upper"] - df["bb_lower"])
    )

    df["price_above_bb_upper"] = (df["close"] > df["bb_upper"]).astype(int)
    df["price_below_bb_lower"] = (df["close"] < df["bb_lower"]).astype(int)

    # ----------------------------
    # Drawdown and breakout indicators
    # ----------------------------
    for window in [30, 60, 90, 180, 365]:
        rolling_high = df["close"].rolling(window).max()
        rolling_low = df["close"].rolling(window).min()

        df[f"drawdown_from_{window}d_high"] = df["close"] / rolling_high - 1
        df[f"distance_from_{window}d_low"] = df["close"] / rolling_low - 1

        df[f"breakout_{window}d_high"] = (
            df["close"] >= rolling_high.shift(1)
        ).astype(int)

        df[f"breakdown_{window}d_low"] = (
            df["close"] <= rolling_low.shift(1)
        ).astype(int)

    # ----------------------------
    # Volume indicators
    # ----------------------------
    df["volume_log"] = np.log1p(df["volume"])

    for window in [7, 14, 30, 60, 90]:
        df[f"volume_sma_{window}"] = df["volume"].rolling(window).mean()
        df[f"volume_z_{window}d"] = rolling_zscore(df["volume_log"], window)
        df[f"volume_change_{window}d"] = df["volume"].pct_change(window)

    df["volume_above_30d_avg"] = (
        df["volume"] > df["volume_sma_30"]
    ).astype(int)

    df["volume_to_volatility"] = (
        df["volume_z_30d"] / df["realized_vol_30d"]
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [11]:
new_df = create_btc_indicators(df)

print(new_df.tail())
print(new_df.shape)

           date       open       high        low      close        volume  \
4293 2026-06-19  62897.516  63568.223  62275.582  63540.836  2.236193e+10   
4294 2026-06-20  63535.809  64307.137  63149.660  64239.676  1.748638e+10   
4295 2026-06-21  64242.324  64506.578  63221.000  63237.539  1.573962e+10   
4296 2026-06-22  63240.789  65544.000  63233.531  63952.105  2.656150e+10   
4297 2026-06-24  63951.508  64163.449  61990.453  62695.230  2.946854e+10   

      log_close  log_return_1d  pct_return_1d  log_return_2d  ...  \
4293  11.059438       0.010193       0.010245      -0.013717  ...   
4294  11.070376       0.010938       0.010998       0.021131  ...   
4295  11.054653      -0.015723      -0.015600      -0.004785  ...   
4296  11.065890       0.011236       0.011300      -0.004487  ...   
4297  11.046041      -0.019849      -0.019653      -0.008613  ...   

      volume_z_30d  volume_change_30d  volume_sma_60  volume_z_60d  \
4293     -0.922258          -0.161481   3.300335e+10

/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_28434/3156723235.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"distance_from_{window}d_low"] = df["close"] / rolling_low - 1
/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_28434/3156723235.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"breakout_{window}d_high"] = (
/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_28434/3156723235.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

* Drop Missing columns

In [12]:
# date কে index বানান
new_df = new_df.set_index('date')
new_df.index = pd.to_datetime(new_df.index)

print(new_df.index[:3])
print(new_df.index.dtype)

DatetimeIndex(['2014-09-17', '2014-09-18', '2014-09-19'], dtype='datetime64[us]', name='date', freq=None)
datetime64[us]


In [13]:
new_df_clean = new_df.dropna()

print(new_df.tail())
print(new_df.shape)

                 open       high        low      close        volume  \
date                                                                   
2026-06-19  62897.516  63568.223  62275.582  63540.836  2.236193e+10   
2026-06-20  63535.809  64307.137  63149.660  64239.676  1.748638e+10   
2026-06-21  64242.324  64506.578  63221.000  63237.539  1.573962e+10   
2026-06-22  63240.789  65544.000  63233.531  63952.105  2.656150e+10   
2026-06-24  63951.508  64163.449  61990.453  62695.230  2.946854e+10   

            log_close  log_return_1d  pct_return_1d  log_return_2d  \
date                                                                 
2026-06-19  11.059438       0.010193       0.010245      -0.013717   
2026-06-20  11.070376       0.010938       0.010998       0.021131   
2026-06-21  11.054653      -0.015723      -0.015600      -0.004785   
2026-06-22  11.065890       0.011236       0.011300      -0.004487   
2026-06-24  11.046041      -0.019849      -0.019653      -0.008613   

    

In [14]:
FEATURE_COLS = [
    # === Returns (relative, scale-free) ===
    'log_return_1d', 'pct_return_1d',
    'log_return_2d', 'pct_return_2d',
    'log_return_3d', 'pct_return_3d',
    'log_return_5d', 'pct_return_5d',
    'log_return_7d', 'pct_return_7d',
    'log_return_14d','pct_return_14d',
    'log_return_21d','pct_return_21d',
    'log_return_30d','pct_return_30d',

    # === MA relative distance (price নয়, distance) ===
    'close_to_sma_7',  'close_to_ema_7',  'sma_7_slope',  'ema_7_slope',
    'close_to_sma_14', 'close_to_ema_14', 'sma_14_slope', 'ema_14_slope',
    'close_to_sma_20', 'close_to_ema_20', 'sma_20_slope', 'ema_20_slope',
    'close_to_sma_50', 'close_to_ema_50', 'sma_50_slope', 'ema_50_slope',
    'close_to_sma_100','close_to_ema_100',
    'close_to_sma_200','close_to_ema_200',

    # === MA Crossover signals ===
    'sma_20_above_50', 'sma_50_above_200', 'ema_20_above_50',

    # === Momentum ===
    'rsi_14', 'rsi_overbought', 'rsi_oversold',
    'macd', 'macd_signal', 'macd_hist', 'macd_bullish',

    # === Volatility ===
    'realized_vol_7d',  'realized_vol_14d', 'realized_vol_21d',
    'realized_vol_30d', 'realized_vol_60d',
    'atr_pct_14',           # atr_14 বাদ — absolute price
    'daily_range_pct',      # daily_range বাদ — absolute price
    'close_position_in_range',

    # === Bollinger Bands (relative only) ===
    'bb_width', 'bb_percent_b',
    'price_above_bb_upper', 'price_below_bb_lower',

    # === Drawdown / Breakout ===
    'drawdown_from_30d_high',  'distance_from_30d_low',
    'breakout_30d_high',       'breakdown_30d_low',
    'drawdown_from_60d_high',  'distance_from_60d_low',
    'drawdown_from_90d_high',  'distance_from_90d_low',
    'breakout_90d_high',       'breakdown_90d_low',
    'drawdown_from_180d_high', 'distance_from_180d_low',
    'drawdown_from_365d_high', 'distance_from_365d_low',

    # === Volume (relative only) ===
    'volume_z_7d',   'volume_change_7d',
    'volume_z_14d',  'volume_change_14d',
    'volume_z_30d',  'volume_change_30d',
    'volume_z_60d',  'volume_change_60d',
    'volume_z_90d',  'volume_change_90d',
    'volume_above_30d_avg',
    'volume_to_volatility',
]

print(f"Total features: {len(FEATURE_COLS)}")

# নিশ্চিত করুন সব column df-এ আছে
missing = [c for c in FEATURE_COLS if c not in new_df.columns]
print(f"Missing columns: {missing}")

Total features: 84
Missing columns: []


In [15]:
new_df.tail()

,open,high,low,close,volume,log_close,log_return_1d,pct_return_1d,log_return_2d,pct_return_2d,...,volume_z_30d,volume_change_30d,volume_sma_60,volume_z_60d,volume_change_60d,volume_sma_90,volume_z_90d,volume_change_90d,volume_above_30d_avg,volume_to_volatility
date,,,,,,,,,,,,,,,,,,,,,
2026-06-19,62897.516,63568.223,62275.582,63540.836,2.236193e+10,11.059438,0.010193,0.010245,-0.013717,-0.013624,...,-0.922258,-0.161481,3.300335e+10,-1.006508,-0.436364,3.417648e+10,-1.132585,0.059338,0,-40.808760
2026-06-20,63535.809,64307.137,63149.660,64239.676,1.748638e+10,11.070376,0.010938,0.010998,0.021131,0.021356,...,-1.510304,-0.355732,3.268724e+10,-1.670355,-0.520310,3.403624e+10,-1.827373,-0.419212,0,-66.281299
2026-06-21,64242.324,64506.578,63221.000,63237.539,1.573962e+10,11.054653,-0.015723,-0.015600,-0.004785,-0.004773,...,-1.657254,-0.435026,3.214395e+10,-1.892718,-0.674375,3.363880e+10,-2.073372,-0.694428,0,-73.556740
2026-06-22,63240.789,65544.000,63233.531,63952.105,2.656150e+10,11.065890,0.011236,0.011300,-0.004487,-0.004477,...,-0.303795,-0.113826,3.191406e+10,-0.362504,-0.341802,3.348747e+10,-0.519748,-0.338962,0,-13.561585
2026-06-24,63951.508,64163.449,61990.453,62695.230,2.946854e+10,11.046041,-0.019849,-0.019653,-0.008613,-0.008576,...,-0.069577,0.432407,3.185880e+10,-0.055865,-0.101136,3.342145e+10,-0.207666,-0.167785,0,-3.098487


In [16]:
# ধাপ ১: আগে target তৈরি করুন
new_df['target'] = new_df['close'].pct_change().shift(-1) * 100
new_df.dropna(subset=['target'], inplace=True)

# ধাপ ২: তারপর FEATURE_COLS বানান
exclude_cols = [
    'date', 
    'target',
]
FEATURE_COLS = [col for col in new_df.columns if col not in exclude_cols]
print(f"Total features: {len(FEATURE_COLS)}")

# ধাপ ৩: তারপর split করুন
new_df.index = pd.to_datetime(new_df.index).tz_localize(None)
split_date   = pd.Timestamp("2025-01-01")
train = new_df[new_df.index < split_date]
test  = new_df[new_df.index >= split_date]
print(f"Train: {len(train)} rows | {train.index[0].date()} → {train.index[-1].date()}")
print(f"Test : {len(test)} rows  | {test.index[0].date()} → {test.index[-1].date()}")

# ধাপ ৪: X, y আলাদা করুন
X_train = train[FEATURE_COLS]
y_train = train['target']
X_test  = test[FEATURE_COLS]
y_test  = test['target']

print(f"\nX_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train mean: {y_train.mean():.4f}%")
print(f"y_test  mean: {y_test.mean():.4f}%")

Total features: 136
Train: 3759 rows | 2014-09-17 → 2024-12-31
Test : 538 rows  | 2025-01-01 → 2026-06-22

X_train: (3759, 136)
X_test : (538, 136)
y_train mean: 0.2079%
y_test  mean: -0.0487%


In [17]:
#new_df["date"] = pd.to_datetime(new_df["date"])
#new_df = new_df.sort_values("date")
#new_df = new_df.set_index("date")

In [18]:
split_date = pd.Timestamp("2025-01-01")

train = new_df[new_df.index < split_date]
test  = new_df[new_df.index >= split_date]


print(f"Train: {len(train)} rows | {train.index[0].date()} → {train.index[-1].date()}")
print(f"Test : {len(test)} rows  | {test.index[0].date()} → {test.index[-1].date()}")

Train: 3759 rows | 2014-09-17 → 2024-12-31
Test : 538 rows  | 2025-01-01 → 2026-06-22


In [19]:
X_train = train[FEATURE_COLS]
y_train = train['target']
X_test  = test[FEATURE_COLS]
y_test  = test['target']

print(f"\nX_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")



X_train: (3759, 136)
X_test : (538, 136)


In [20]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler


In [21]:
# Scale
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


In [22]:
# Model
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train_sc, y_train)
preds = xgb.predict(X_test_sc)

# Metrics
mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)
dir_acc = ((preds > 0) == (y_test > 0)).mean() * 100

print(f"MAE             : {mae:.4f}%")
print(f"RMSE            : {rmse:.4f}%")
print(f"R²              : {r2:.4f}")
print(f"Direction Acc   : {dir_acc:.2f}%")

# Feature Importance Top 15
import pandas as pd
importance = pd.Series(xgb.feature_importances_, index=FEATURE_COLS)
print("\n🔑 Top 15 Important Features:")
print(importance.nlargest(15).round(4).to_string())

MAE             : 1.6568%
RMSE            : 2.3426%
R²              : -0.0066
Direction Acc   : 50.19%

🔑 Top 15 Important Features:
sma_50                     0.0150
close_to_ema_20            0.0143
close_to_sma_7             0.0143
log_return_14d             0.0138
macd_signal                0.0133
ema_200_slope              0.0133
bb_upper                   0.0133
pct_return_3d              0.0132
volume_z_7d                0.0124
distance_from_180d_low     0.0123
close_to_sma_20            0.0122
ema_50_slope               0.0121
close_to_ema_30            0.0121
sma_7                      0.0120
drawdown_from_365d_high    0.0119


In [23]:
print(new_df['target'].describe())
print(new_df['target'].head(10))

count    4297.000000
mean        0.175791
std         3.486211
min       -37.169538
25%        -1.262676
50%         0.105088
75%         1.587105
max        25.247168
Name: target, dtype: float64
date
2014-09-17   -7.192555
2014-09-18   -6.984262
2014-09-19    3.573491
2014-09-20   -2.465860
2014-09-21    0.835212
2014-09-22    8.364748
2014-09-23   -2.888082
2014-09-24   -2.748313
2014-09-25   -1.736990
2014-09-26   -1.212833
Name: target, dtype: float64


In [24]:
print(new_df['target'].describe().round(4))
print(new_df[['close', 'target']].head(10))

count    4297.0000
mean        0.1758
std         3.4862
min       -37.1695
25%        -1.2627
50%         0.1051
75%         1.5871
max        25.2472
Name: target, dtype: float64
              close    target
date                         
2014-09-17  457.334 -7.192555
2014-09-18  424.440 -6.984262
2014-09-19  394.796  3.573491
2014-09-20  408.904 -2.465860
2014-09-21  398.821  0.835212
2014-09-22  402.152  8.364748
2014-09-23  435.791 -2.888082
2014-09-24  423.205 -2.748313
2014-09-25  411.574 -1.736990
2014-09-26  404.425 -1.212833
